## Step 4: **Build the RAG Pipeline Components**

In [4]:
# ========================================================
#  COMPONENT 1 - Document Ingestion
# ========================================================

class DocumentIngester:
    """Loads and chunks smart contract documents (PDF / DOCX / TXT)."""

    def __init__(self, chunk_size: int = 800, chunk_overlap: int = 150):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    def load(self, file_path: str) -> List[Document]:
        path = Path(file_path)
        ext  = path.suffix.lower()

        if ext == ".pdf":
            loader = PyMuPDFLoader(str(path))
            docs = loader.load()
        elif ext in (".docx", ".doc"):
            loader = Docx2txtLoader(str(path))
            docs = loader.load()
        elif ext in (".txt", ".text"):
            content = path.read_text(encoding="utf-8", errors="ignore")
            docs = [Document(page_content=content, metadata={})]
        else:
            raise ValueError(f"Unsupported file type: '{ext}'. Use PDF, DOCX, or TXT.")

        for doc in docs:
            doc.metadata.update({"source": path.name, "file_type": ext.lstrip(".")})
        return docs

    def chunk(self, documents: List[Document]) -> List[Document]:
        chunks = self.splitter.split_documents(documents)
        for i, c in enumerate(chunks):
            c.metadata["chunk_id"] = i
        return chunks

    def ingest(self, file_path: str) -> List[Document]:
        """Full pipeline: load then chunk."""
        return self.chunk(self.load(file_path))


print("\u2705 DocumentIngester defined.")

✅ DocumentIngester defined.
